# 05. Warmed-up Latency & Peak GPU/CPU Memory Profiling
Runs 5 warm-up iterations followed by 30 synchronized measured runs using `torch.cuda.synchronize()`.
Measures prefill, decoding latency, tokens/sec, and peak GPU VRAM allocation.

In [1]:
# Cell 1: Environment Setup & Thư viện
!pip install -q bitsandbytes accelerate transformers pandas scipy
import os, sys, json, time, torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print('✅ PyTorch Version:', torch.__version__)
print('✅ CUDA Available:', torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.7 MB/s eta 0:00:00
✅ PyTorch Version: 2.10.0+cu128
✅ CUDA Available: True


In [2]:
# Cell 2: Dataset Search & Path Verification
possible_paths = [
    '/kaggle/input/datasets/tunthanh66/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/trungkiennnn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/tunthanh66/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json',
    './vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    search_root = '/kaggle/input' if os.path.exists('/kaggle/input') else '.'
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if file.endswith('.json') and ('halueval' in file.lower() or '15k' in file.lower() or 'medical' in file.lower() or 'vnese' in file.lower()):
                data_path = os.path.join(root, file)
                break
        if data_path: break

print('✅ Resolved Dataset Path:', data_path)
with open(data_path, 'r', encoding='utf-8') as f: full_dataset = json.load(f)
test_data = full_dataset[-500:]
print('Total dataset size:', len(full_dataset), '| Test size:', len(test_data))
assert len(test_data) == 500, 'Test size must be 500'


✅ Resolved Dataset Path: /kaggle/input/datasets/tunthanh66/vnese-data/vietnamese_medical_halueval_15k_specialized.json
Total dataset size: 14700 | Test size: 500


In [3]:
# Cell 3: Model & Tokenizer Load (Qwen2.5-7B-Instruct)
model_id = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.eval()
print('✅ Tokenizer & Model Loaded Successfully!')


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Tokenizer & Model Loaded Successfully!


In [4]:
# Cell 4: 5 Warm-up Runs & 30 Synchronized Measured Runs
sample_input = tokenizer("Xin chào bác sĩ, thuốc Paracetamol có dùng được cho phụ nữ mang thai không?", return_tensors="pt").to(model.device)
torch.cuda.reset_peak_memory_stats()

print('⚡ Performing 5 Warm-up Runs...')
for _ in range(5):
    with torch.no_grad():
        model.generate(**sample_input, max_new_tokens=50, do_sample=False)

print('⚡ Running 30 Synchronized Latency Measurements...')
latencies, tokens_per_sec = [], []
for _ in range(30):
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**sample_input, max_new_tokens=200, do_sample=False)
    torch.cuda.synchronize()
    lat = time.time() - t0
    latencies.append(lat)
    n_toks = out[0].shape[0] - sample_input.input_ids.shape[1]
    tokens_per_sec.append(n_toks / lat)

peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
reserved_vram_mb = torch.cuda.max_memory_reserved() / (1024 * 1024)

res_summary = {
    'mean_latency': float(np.mean(latencies)),
    'std_latency': float(np.std(latencies)),
    'median_latency': float(np.median(latencies)),
    'p95_latency': float(np.percentile(latencies, 95)),
    'tokens_per_sec_mean': float(np.mean(tokens_per_sec)),
    'peak_vram_allocated_mb': float(peak_vram_mb),
    'peak_vram_reserved_mb': float(reserved_vram_mb)
}

os.makedirs('results', exist_ok=True)
with open('results/latency_memory_summary.json', 'w', encoding='utf-8') as f:
    json.dump(res_summary, f, indent=2, ensure_ascii=False)
pd.DataFrame([res_summary]).to_csv('results/latency_memory_summary.csv', index=False)

print('========================================================================')
print(f'✅ Warmed-up Mean Latency: {res_summary["mean_latency"]:.4f}s ± {res_summary["std_latency"]:.4f}s')
print(f'✅ Median Latency: {res_summary["median_latency"]:.4f}s, P95: {res_summary["p95_latency"]:.4f}s')
print(f'✅ Generation Throughput: {res_summary["tokens_per_sec_mean"]:.2f} tokens/sec')
print(f'✅ Peak Allocated GPU Memory: {res_summary["peak_vram_allocated_mb"]:.2f} MB')
print(f'✅ Peak Reserved GPU Memory: {res_summary["peak_vram_reserved_mb"]:.2f} MB')
print('========================================================================')
assert res_summary['mean_latency'] > 0, 'Latency must be greater than 0'
print('✅ Automated Latency & Memory Assertions Passed 100%!')


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


⚡ Performing 5 Warm-up Runs...
⚡ Running 30 Synchronized Latency Measurements...
✅ Warmed-up Mean Latency: 15.2519s ± 0.2340s
✅ Median Latency: 15.2262s, P95: 15.6916s
✅ Generation Throughput: 13.12 tokens/sec
✅ Peak Allocated GPU Memory: 1409.37 MB
✅ Peak Reserved GPU Memory: 1666.00 MB
✅ Automated Latency & Memory Assertions Passed 100%!
